

# 🧭 Using Futures to Isolate Sources of Risk and Return
<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Derivatives-Futures-Chapter-Three/blob/master/colab/Colab_span_risk_return.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Derivatives-Futures-Chapter-Three/master?urlpath=lab/tree/notebooks/span_risk_return.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Derivatives-Futures-Chapter-Three/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
The previous chapter used the E-mini S&P 500 to demonstrate how to synthetically control the risk of an investment in the S&P 500. That demonstration highlighted the variability of arbitrage bounds. Our intent was and is to apply these contracts to enhance the risk-return trade-off. Although we can discuss the theory, to truly grasp the concepts we have to interact with the data. This chapter goes beyond the S&P 500, looking at other contracts that can be used to span the available risk-return trade-off. Our first order of business is to identify the relevant futures contracts for the exercise.

## 🧰 The Toolkit: Liquid Stock Index Contracts

The U.S. stock index contracts traded on the CME are shown in the table below. All of these contracts are cash-settled and settle on the third Friday of the contract month. The most relevant data here is the Daily Volume column.

| Contract &emsp;&emsp;&emsp;&emsp; | &emsp;Ticker&emsp;&emsp;&emsp; | &emsp;Multiplier &emsp;&emsp;&emsp; | &emsp;&emsp;&emsp; Daily Volume &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; |
| :--- | :---: | :---: | ---: |
| E-mini S&P 500 | ES | 50 | 1.2-1.3 (1,000,000) |
| E-mini Nasdaq 100 | NQ | 20 | 500-800 (1,000) |
| E-mini Russell 2000 | RTY | 50 | 150-250 (1,000) |
| E-mini Dow Jones | YM | 5 | 100-200 (1,000) |
| E-mini S&P MidCap 400 | EMD | 100 | 4-7 (1,000) |

As you might expect, the E-mini S&P 500 has the highest volume. With the exception of the E-mini S&P MidCap 400, these contracts are highly liquid. Because the volume of the MidCap 400 contract is more than an order of magnitude lower than the others, it's excluded from our toolkit. The available tools are the four highly liquid contracts: ES, NQ, RTY, and YM.

## 🎯 The Targets: Capturing the Risk of Benchmark Portfolios

It makes sense to structure an investment strategy around the attractiveness of different risks—an approach identified as factor investing. Often, there are no liquid futures contracts that directly apply to specific market factors.

Our goal is to demonstrate how to use our four liquid futures contracts to **span** (synthetically replicate) the risk-return profiles of five target benchmark portfolios:

* **The S&P 500:** Generally described as the generic stock market risk—the benchmark used to estimate the beta of individual stocks and portfolios.
* **The S&P MidCap 400:** Generally described as the intermediate market risk—the benchmark used to capture the performance of mid-sized companies bridging the gap between early growth and mature stability.
* **The S&P SmallCap 600:** Generally described as the higher-volatility market risk—the primary benchmark used to isolate the risk and return premium associated with smaller, domestically sensitive companies.
* **S&P 500 Growth:** Constructed from the S&P 500, this index focuses on companies projected to grow their earnings and revenue at a significantly faster rate than the broader market. Growth stocks typically trade at higher valuations (such as elevated price-to-earnings ratios) and tend to reinvest their profits back into the business to fuel further expansion. These selection metrics result in an index heavily weighted toward large-cap stocks.
* **S&P 500 Value:** Also constructed from the S&P 500, this index comprises companies trading at a discount relative to their underlying fundamentals, such as earnings, book value, or cash flow. Value stocks are often mature, established businesses in cyclical industries that return capital to shareholders through steady dividends. The value metrics tend to skew the index away from the largest mega-cap stocks.

## 🧪 The Test
We'll measure the success of the exercise by comparing how well the replicating futures positions match the actual returns of the portfolios. Our comparison will not be in-sample—a method that fails to reflect real-world trading. Instead, we'll judge success by how accurately rolling estimates of the spanning futures contracts forecast future S&P index returns

## ♾️ Universal Application
 It is worth noting that this technique isn't limited to these five benchmarks. Once you understand the mechanics of return spanning, you can use these exact same liquid contracts to synthesize a wide variety of other market factors, sectors, or custom portfolios. It's a plug and play exercise!

## 📦 Getting Functions from financial_quant package



## ⚙️ Environment Setup: A Quick Note on the Tools

In the first code cell below, we are going to load our standard data libraries (`pandas` and `numpy`), along with two heavy-duty Python libraries you might not have used yet: `seaborn` and `statsmodels`. If you look up their official documentation, they use a lot of intimidating academic jargon. **Don't overthink them.**

* 🗺️ **Seaborn (`sns`):** For our purposes, this is just an automatic color-coding tool. Later, we will calculate a correlation matrix to see how our contracts relate to each other—the stronger the correlation, the harder it is to synthesize risk. Instead of a grid of decimals, Seaborn applies a color scale to those numbers, creating a **heatmap** highlighting the stronger and weaker correlations.
* 🧮 **Statsmodels (`sm`):** We will use this as a diagnostic tool. It calculates Ordinary Least Squares (OLS) regressions and prints highly detailed summary statistics.

**One catch:** Because `statsmodels` generates such massive reports, we'll measure the success of the exercise by comparing how well the replicating futures positions match the actual returns of the portfolios. Our comparison will not be in-sample—a method that fails to reflect real-world trading. Instead, we'll judge success by how accurately rolling estimates of the spanning futures contracts forecast future S&P index returns.

*Think of `statsmodels` as a luxury car with a dashboard full of diagnostic screens—perfect for deeply analyzing our initial benchmark. Think of `linalg` as a stripped-down race car engine—no pretty reports, just pure speed for our rolling estimates.*

**⬇️ Run the cell below to load the tools and set up the environment.**

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;"><span style="font-size: 1.4em;">⚡🔌</span> Professional Packaging: How GitHub Installations Work</b></summary>
<br>
<p><b>The Logic:</b><br>
Usually, Python looks for modules as <code>.py</code> files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library updater script.</p>

<p><b>The Workflow:</b></p>
<ol>
<li><b>Fetch & Build:</b> The <code>urllib</code> script downloads and executes <code>fq_updater.py</code>, which verifies local dependency versions (like locking Pandas to 2.2.3) and installs or updates the <code>financial_quant</code> package in the background.</li>
<li><b>Import:</b> <code>fq = import_financial_quant()</code> loads the verified package into your notebook's memory and assigns it the quick alias <code>fq</code>.</li>
<li><b>Routing:</b> Behind the scenes, a special file called <code>__init__.py</code> acts as the package's "front door." It automatically gathers complex tools from deeply nested folders (like our fixed-income models and chart visualizers) and serves them up directly to the surface.</li>
<li><b>Execute:</b> You don't have to worry about where the files live. You just type <code>fq.create_arb_vix()</code> or <code>fq.plot_contract_arb_band()</code>, and Python immediately knows where to route the request.</li>
</ol>

<p><b>Why do this?</b><br>
The most important reason is to ensure these notebooks are environment-agnostic—whether you are using Jupyter Lab, Google Colab, or classic Jupyter Notebooks. Using the <code>financial_quant</code> package allows us to align all software versions and interpolation rules across machines. The package installs and manages the libraries; you only have to import them.</p>
</details>

<br>

## ⚙️ Environment Setup: A Quick Note on the Tools

In the first code cell below, we are going to load our standard data libraries (`pandas` and `numpy`), along with two heavy-duty Python libraries you might not have used yet: `seaborn` and `statsmodels`. If you look up their official documentation, they use a lot of intimidating academic jargon. **Don't overthink them.**

* 🗺️ **Seaborn (`sns`):** For our purposes, this is just an automatic color-coding tool. Later, we will calculate a correlation matrix to see how our contracts relate to each other—the stronger the correlation, the harder it is to synthesize risk. Instead of making you stare at a giant grid of decimals, Seaborn simply applies a color scale to those numbers, creating a **heatmap** where the strongest correlations instantly pop out.
*   🧮 **Statsmodels (`sm`):** We will use this as a diagnostic tool. It runs Ordinary Least Squares (OLS) regressions and prints out a beautiful, highly detailed summary report so we can read exactly how well our futures replicate our targets.

**One catch:** Because `statsmodels` generates such massive reports, it is computationally heavy and slow. Later, when we need to run these regressions repeatedly over rolling time windows, `statsmodels` would take forever. We avoid the computational drag by switching gears and using a built-in linear algebra tool (`linalg.solve`)—the exact same tool we used to estimate discrete values of the present value function in *Chapter Three of the Term Structure volume*—to run the same math at lightning speed!

*Think of `statsmodels` as a luxury car with a dashboard full of diagnostic screens—perfect for deeply analyzing our initial benchmark. Think of `linalg` as a stripped-down race car engine—no pretty reports, just pure speed for our rolling estimates.*

**⬇️ Run the cell below to load the tools and set up the environment.**

In [ ]:
# ==========================================
# CELL 1: ENVIRONMENT SETUP & IMPORTS
# Run this cell once to configure the notebook
# ==========================================

# 1. Fetch and install custom financial quant package
import urllib.request
updater_url = "https://raw.githubusercontent.com/PatrickJHess/quant_repo/master/fq_updater.py"
exec(urllib.request.urlopen(updater_url).read().decode('utf-8'))
fq = import_financial_quant() # Alias the package as 'fq'

# 2. Import standard data science libraries
import pandas as pd
import numpy as np

# 3. Import diagnostic and visualization libraries
import seaborn as sns
import statsmodels.api as sm
from matplotlib import pyplot as plt

## 🚰 The Data Pipeline: Fueling the Models

Before we can run our regressions and build our heatmaps, we need to gather our raw materials. Our custom `fq` package handles the heavy lifting, pulling from three distinct data pipelines to assemble everything we need.

Here is where our data comes from and why we need it:

*   **The Toolkit (`massive`):** We pull the daily historical prices for our four highly liquid futures contracts (ES, NQ, RTY, YM) directly from the `massive` futures database. This represents the right side of our equation (our explanatory matrix or $X$ in the `statsmodel`)—the tools we will use to replicate risk.
*   **The Targets (`static_futures_data`):** We fetch the total return series for our five benchmark indices (S&P 500, MidCap, SmallCap, Growth, and Value). These represent the left side of our equation (our $y$ vectors in the `statsmodek`)—the risk profiles we are trying to span.
*   **The Base Rate (`fred`):** To properly evaluate risk and run factor regressions, we cannot just look at raw performance. We need to calculate **excess returns**—how much an asset earned *above* the base financing rate. We use our `fred` module to pull the Secured Overnight Financing Rate (SOFR) directly from the Federal Reserve Economic Data (FRED) database to serve as our base rate.

**⬇️ Run the cell below to instantiate these data connections.**

In [ ]:
# 1. Base Rate Pipeline: Fetches SOFR data to calculate excess returns
fred_data = fq.FredReader()

# 2. Toolkit Pipeline: Fetches our liquid futures data (our X matrix)
# Note: 'calls_per_minute=5' ensures we don't overwhelm the database and get rate-limited!
massive_data = fq.MASSIVEReader(calls_per_minute=5)

# 3. Targets Pipeline: Loads the benchmark index total returns (our y vectors)
static_futures_data = fq.static_futures_data

## 🔄 Building a Continuous Futures Data Time Series: The "Roll"

Unlike stocks, futures contracts have expiration dates. Because an individual equity index contract is only highly liquid for about three months, we cannot just download a single contract to get a year and a half of data.

As you might recall from the previous notebook, we solve this by creating a continuous time series—stitching together quarterly contracts. To maintain our market exposure, we simulate "rolling" our position: approximately eight days before the active contract expires, we sell it and buy the next quarter's contract.

In the code below, we do this for the period spanning roughly March 2025 through October 2026. You will see us use standard industry month codes appended with the year:
*   **H:** March
*   **M:** June
*   **U:** September
*   **Z:** December

**(Example: 'M5' is the June 2025 contract; 'Z6' is the December 2026 contract).**


## 🛠️ Building our "Plug-and-Play" Data Fetcher

To keep our code clean and reusable, we are first going to define a helper function. Give it a delivery month and a date range, and it will automatically fetch the daily settlement prices for all four of our toolkit contracts (ES, RTY, NQ, YM) and neatly group them into one table.

The function includes a console management option. The underlying data fetcher prints a status line for every single contract it checks. This is a useful feature when you are accessing data for a few series. But here we are accessing four months for seven different contract months—twenty-eight notices of success are a bit much. Assigning `True` to `clear_logs` and `wait_to_clear parameters`suppresses the output.

Because we built this as a standalone function, you can easily copy and paste this cell into future projects whenever you need to pull liquid futures data!

---

<details>
<summary> <b>💡Example Of `clear_ouput`</b></summary>
<br>

```python
import time
from IPython.display import clear_output

print("Step 1: Loading...")
time.sleep(1)

# wait=True prevents a blank white flash between messages
clear_output(wait=True)
print("Step 2: Done!")

# Output
# Step 2: Done!
```
</details>



In [ ]:
# ==========================================
# HELPER FUNCTION: Fetch & group toolkit contracts
# ==========================================
def futures_data(date_id, start, end,
                 clear_logs=False,
                 wait_to_clear=False): #By default all successful accessed displayed
    """
    Fetches the settlement prices for our 4 liquid futures contracts
    for a specific delivery month and date range, then combines them.
    """
    # Import clear_output function from Ipython.Display
    from IPython.display import clear_output

    # 1. Fetch the S&P 500 (ES) data and initialize the DataFrame
    df = massive_data.get_futures_data('ES' + date_id, start, end)
    df_settle = df[['settlement_price']].copy().rename(columns={'settlement_price': 'SP500_FUT'})

    # 2. Loop through the remaining 3 toolkit contracts (Small Cap, Growth, Value)
    for symbol, column in zip(['RTY', 'NQ', 'YM'],
                              ['RUSS_FUT', 'NSDQ_FUT', 'DJIA_FUT']):

        # Fetch data, extract just the settlement price, and rename the column
        df = massive_data.get_futures_data(symbol + date_id, start, end)
        df_temp = df[['settlement_price']].copy().rename(columns={'settlement_price': column})

        # Merge this contract's data side-by-side with the others
        df_settle = pd.concat([df_settle, df_temp], axis=1)
    if clear_logs:
        clear_output(wait=wait_to_clear)
    return df_settle

### 🏗️ Fetching Seven Quarters of Data
Before stitching our continuous time series together, we need to download the raw data: four spanning futures contracts for each of the seven quarters. All in, we'll pull twenty-eight individual contracts.

In [ ]:
# ==========================================
# DATA PIPELINE: Stitching the quarters together
# ==========================================
# Notice how the end date of one contract matches the start date of the next.
# This represents the day we "roll" our exposure forward.

df_jun5 = futures_data('M5', '2025-03-13', '2025-06-12',
                       clear_logs=True,
                       wait_to_clear=True) # June 2025 contract
df_sep5 = futures_data('U5', '2025-06-12', '2025-09-11',
                       clear_logs=True,
                       wait_to_clear=True) # Sept 2025 contract
df_dec5 = futures_data('Z5', '2025-09-11', '2025-12-11',
                       clear_logs=True,
                       wait_to_clear=True) # Dec 2025 contract
df_mar6 = futures_data('H6', '2025-12-11', '2026-03-12',
                       clear_logs=True,
                       wait_to_clear=True) # Mar 2026 contract
df_jun6 = futures_data('M6', '2026-03-12', '2026-06-11',
                       clear_logs=True,
                       wait_to_clear=True) # June 2026 contract
df_sep6 = futures_data('U6', '2026-06-11', '2026-09-10',
                       clear_logs=True,
                       wait_to_clear=True) # Sept 2026 contract

# Display the results for the quarterly contract: December 2026
df_dec6 = futures_data('Z6', '2026-09-10', '2026-10-07',
                       clear_logs=False,
                       wait_to_clear=False) # Dec 2026 contract

### 🧩 Stacking the Blocks: Verifying the Roll

Now that we have seven separate quarters of data, we need to stack them vertically into a single, continuous dataset. We do this using Pandas' `concat` function.

It always pays to verify a data merge. Stitching together time series data is exactly where unexpected bugs tend to hide.

To prove our data is clean, we are going to inspect the "seams." In the code below, we isolate the exact dates where our contracts transition (the roll dates). Displaying these specific days allows us to verify that the hand-off from one contract to the next was perfectly executed.

---


<details>
<summary><b>📝 A Quick Example: Stacking with pd.concat</b></summary>
<br>
When you pass a list of DataFrames to <code>pd.concat()</code> without specifying an axis, pandas defaults to <code>axis=0</code>. This stacks the DataFrames vertically, matching columns by name.

```python
import pandas as pd
df_sep = pd.DataFrame({'Forward Rate': [4.8]}, index=['2025-09-17'])
df_dec = pd.DataFrame({'Forward Rate': [4.9]}, index=['2025-12-17'])

# Stacking them together
pd.concat([df_sep, df_dec])

# Result:
#             Forward Rate
# 2025-09-17           4.8
# 2025-12-17           4.9
```
</details>


In [ ]:
# 1. Stack all the quarterly dataframes vertically into one continuous dataset
df_futures_raw = pd.concat([df_jun5, df_sep5, df_dec5, df_mar6, df_jun6, df_sep6, df_dec6])

# 2. Extract the exact "seam" dates where the contracts transition
sep5_roll = df_futures_raw.loc['2025-06-12']
dec5_roll = df_futures_raw.loc['2025-09-11']
mar6_roll = df_futures_raw.loc['2025-12-11']
jun6_roll = df_futures_raw.loc['2026-03-12']
sep6_roll = df_futures_raw.loc['2026-06-11']
dec6_roll = df_futures_raw.loc['2026-09-10']

# 3. Group and display these specific dates to verify the roll was successful
pd.concat([sep5_roll, dec5_roll, mar6_roll, jun6_roll, sep6_roll, dec6_roll])

## ⚠️ The Danger of the Roll Gap

Look closely at the output above. You should notice that every roll date has **two rows**, and the prices are **different**. For example, on June 12, 2025, our expiring June S&P contract settled at 6049.50, but the new September contract settled at 6103.00.

**Why are the prices different?**

This price gap represents the "cost of carry" highlighted in the previous notebook: the difference between the base financing rate and the dividend yield of the underlying index over those extra three months.

**Avoid Mixing Contracts:**

If we were to just run a standard percentage change calculation (`pct_change()`) straight down the raw price columns, the jump from the old contracts to the new contract would register as a massive, fake daily return.



## 🧹 Safe Returns and Defensive Data Cleaning

To avoid generating artificial returns across our roll gaps, we will calculate the percentage change (pct_change()) on each quarterly block individually. Only after these returns are calculated safely will we stitch them together.

Next, we apply a best practice of quantitative finance: **defensive programming**.Even though these four futures contracts are highly liquid and should have prices for every single trading day, real-world data is messy. Exchange feed outages, mismatched holiday schedules, or database glitches can occasionally drop a missing value (`NaN`) into our dataset. A single NaN will break our logic. Appending `.dropna()` cleanly removes any affected rows from `df_futures`. Once the data is scrubbed, we are good to go.

---


<details>
<summary><b>📝 A Quick Example: dropna()</b></summary>
<br>

Create a DataFrame with a missing value (NaN)
```python
import pandas as pd
df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Age': [25, np.nan, 35]})

# Drop rows with missing values
result = df.dropna()

print(result)

# Result:

#     Name    Age
# 0   Alice   25.0
#2    Charlie 35.0
```
</details>


In [ ]:
df_futures=df_futures_raw.dropna()
df_futures.head()

### 🗺️ Visualizing Spanning Ability: The Correlation Heatmap

Now we need to look at how our toolkit contracts relate to one another. Our goal is spanning—using these liquid futures to replicate specific risk profiles. If all four contracts are perfectly correlated, they act like the exact same tool, making it impossible to isolate different types of risk. We want to see how distinct their movements actually are.

**The Math Behind the Map:**

To measure this, we use the Pandas `.corr()` method. By default, this calculates the **Pearson correlation coefficient** measuring the linear relation between the futures. If two contracts have a correlation too close to 1.0, their risk exposure is alsmost the same. In regression math, this is called multicollinearity—a condition that can produce unreliable results, as one combination is almost as good as another.

**An Elegant Pandas Trick for the Roll Gap:**

Earlier, we saw that our stitched raw data (`df_futures_raw`) has duplicate dates on the days where the contracts roll. To prevent these gaps from generating artificial price jumps, we can use a highly efficient Pandas trick: `duplicated()`. Our first step makes the basic percentage change calculations; the second, catches the roll date with `duplicated()`.

1. Run `pct_change(`) down the entire raw, stitched column. This creates a valid return on the first instance of the roll date, but a massive, fake return on the second instance (the gap).

2. Apply `~df.index.duplicated(keep='first')`. This tells Pandas to look at the dates, find any duplicates, and keep only the first one. It perfectly slices out the fake gap returns while leaving the valid daily returns intact.

**A Warning on Data Alignment:**

Right now, this dataset only contains our futures contracts. In the next steps, we will bring in our target indices and the SOFR base rate. Because these datasets come from different sources, there is no guarantee their trading days and holidays will match perfectly. We will have to explicitly align and clean the data again once everything is merged.


---


<details>
<summary><b>📝 A Simple Example Of duplicated()</b></summary>
<br>


```python
import pandas as pd

# Create a DataFrame with duplicated index labels
df = pd.DataFrame({'Value': [10, 20, 30, 40]}, index=[0, 1, 0, 1])

# keep='first' is the default behavior and must be a lowercase string
result = df[~df.index.duplicated(keep='first')]
print(result)

# Result:
#            Value
# 0           10
# 1           20
```
</details>

In [ ]:
# 1. Calculate returns and remove the fake roll-gap returns
df_fut_pct_dup = df_futures_raw.pct_change(fill_method=None)

# Keep only the first instance of any duplicated date
df_fut_pct = df_fut_pct_dup[~df_fut_pct_dup.index.duplicated(keep='first')]

# Calculate the Pearson correlation matrix
corr_matrix = df_fut_pct.corr()

# 2. Plot the correlation heatmap using Seaborn
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,       # Display correlation values inside cells
    fmt='.2f',        # Format numbers to 2 decimal places
    cmap='coolwarm',  # Color palette (red = positive, blue = negative)
    vmin=0.5,
    vmax=1.0,         # Set colorbar range between 0.5 and 1.0
    square=True,      # Force square cells
    linewidths=0.5,   # Add spacing grid lines between cells
    cbar_kws={'shrink': 0.8},
)

plt.title('Toolkit Contract Return Correlation Matrix', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

### 📊 Interpreting the Heatmap: High Correlation, but "Room to Roam"

Looking at the matrix above, the first thing that jumps out is that **all four contracts are strongly positively correlated**—ranging from **0.78** (Nasdaq vs. Dow) up to **0.95** (S&P 500 vs. Nasdaq). This makes intuitive economic sense: when equity markets experience broad macro tailwinds or headwinds, broad-market equity indices generally move in the same direction.

The key takeaway is **unexplained variance**:

* **The Core Anchor (0.95):** S&P 500 (`SP500_FUT`) and Nasdaq (`NSDQ_FUT`) share a tight 0.95 correlation due to heavy mega-cap tech overlap.
* **Distinct Risk Profiles (0.78, 0.86):** Contracts like `RUSS_FUT` (Small Cap) and `DJIA_FUT` (Value/Cyclicals) drop into the 0.78 to 0.86 correlation range against Tech and Large Caps.

**Why this matters for spanning:**

A correlation of 0.80 means that roughly $36\%$ of the daily return variance between those assets ($1 - 0.80^2 \approx 0.36$) is completely unlinked. That remaining percentage provides critical **"room to roam."** Because these contracts do not move in perfect lockstep ($1.00$), OLS estimates can isolate unique factor loadings (e.g., tech growth vs. small-cap volatility vs. value stability) to synthesize targeted index benchmarks without triggering total mathematical collinearity. In short, the data should have a tale to tell.

<br>

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;"><span style="font-size: 1.4em;">💡</span> Deep Dive: The Amount of Data</b></summary>
<br>

You can think about data in terms of how rich it is relative to everything else, but there is another side. The more data you have, the more chances you have of detecting a pattern. **The number of observations matters.**

In empirical finance, having a large number of trading observations ($N$) directly shrinks the standard errors of our parameter estimates: you learn more with more chances to learn. When evaluating spanning and regression models:

1. **Statistical Power:** More daily return observations make our estimates of correlation ($\rho$) and regression coefficients ($\beta$) significantly more precise, ensuring that measured differences between contracts aren't just short-term statistical noise.
2. **Regime Coverage:** A longer time series captures a broader range of market conditions—volatility spikes, trending bull phases, and range-bound chop—giving us confidence that our replication strategy holds across different market regimes.

</details>

<br>

## 🎯 Assembling the Target Benchmarks ($y$ Vectors)

In the math of a regression equation, the futures contracts are the $X$ matrix-the independent variables doing the explaining. The targets we want to explain are the $y$. Now that our toolkit futures ($X$ matrix) is structured and verified, we turn to the left side of our equation ($y$): the benchmark portfolios we intend to synthesize.

We pull five core S&P Total Return benchmarks:
*   **S&P 500 Total Return (`SP500_tr`):** The broad-market large-cap anchor.
*   **S&P MidCap 400 Total Return (`SPMID_tr`):** Mid-cap market exposure.
*   **S&P 500 Growth Total Return (`SPGRW_tr`):** Large-cap companies with strong growth characteristics.
*   **S&P SmallCap 600 Total Return (`SPSML_tr`):** Small-cap market exposure.
*   **S&P 500 Value Total Return (`SPVAL_tr`):** Large-cap companies trading at value multiples.

**Why Total Return (`_tr`)?**
Unlike price-return indices that only track headline price fluctuations, **Total Return Indices (TRI)** account for the reinvestment of cash dividends. Because a real portfolio holder collects those dividends, synthetic replication must match total economic performance, not just price changes.

**Data Alignment via Inner Join:**
In the loop below, we fetch each series from `static_futures_data`, standardize the dates using `pd.to_datetime()`, and progressively merge them using an **inner join** (`how='inner'`). This guarantees that only dates where all five benchmark indices traded are retained, ensuring clean, gap-free target vectors.

**⬇️ Run the cell below to fetch and align the target index series.**

In [ ]:
# 1. Define the four additional S&P total return benchmarks to fetch
index_names = ['SPMID', 'SPGRW', 'SPSML', 'SPVAL']

# 2. Fetch the baseline S&P 500 Total Return index and set up the master DataFrame
df_tri = static_futures_data('SPTR_Index')
df_tri.columns = ['SP500_tr']
df_tri.index = pd.to_datetime(df_tri.index)

# 3. Fetch each benchmark, standardize the timestamp, and align using an inner join
for name in index_names:
    column_name = name + '_tr'
    df_temp = static_futures_data(name + '_Index')
    df_temp.columns = [column_name]
    df_temp.index = pd.to_datetime(df_temp.index)

    # Inner join guarantees only dates common to all indices are kept
    df_tri = df_tri.join(df_temp, how='inner')

# 4. Preview the most recent records to confirm clean date alignment
df_tri.tail()

## 🔗 Merging the $X$ and $y$ Variables: The Left Join Trick

Now we need to combine our futures data (our $X$ matrix) with our target benchmark indices (our $y$ vectors).

**The Alignment Challenge:**
Remember that our stitched futures dataset (`df_futures`) contains **duplicate dates** on the days we roll our contracts. Our benchmark dataset (`df_tri`), however, only has **one row per trading day**.

**The Left Join Solution:**
To solve this misalignment, we execute a left join using `df_futures` as the anchor table. Pandas automatically maps the single matching benchmark value from `df_tri` onto every matching timestamp in `df_futures`—duplicating the benchmark value across roll-gap rows. This keeps our target $y$ vectors and predictor $X$ matrix perfectly aligned for the return calculations.


---


<details>
<summary><b>📝 A Quick Example Left Join:how='left'</b></summary>
<br>

```python
import pandas as pd
# Left DataFrame with a duplicated index (e.g., duplicate 'A')
left_df = pd.DataFrame({
    'Name': ['Alice', 'Alex', 'Bob']
}, index=['A', 'A', 'B'])

# Right DataFrame with missing data (NaN)
right_df = pd.DataFrame({
    'Dept': ['Analytics', 'IT']
}, index=['A', 'B'])

# Perform a left join on the indexes
result = left_df.join(right_df, how='left')

print(result)

# Result:
#      Name    Dept
# A   Alice   Analytics
# A    Alex   Analytics
# B   Bob            IT
```
</details>


In [ ]:
# Execute a left join to broadcast the target index values across the roll date duplicates
df_futures_cash = df_futures.join(df_tri, how='left')

## 🏦 Fetching the Base Financing Rate (SOFR)

To evaluate our spanning strategy properly, we need to calculate **excess returns**—the return generated *above* the financing rate. In modern institutional finance, our base financing rate is SOFR (the Secured Overnight Financing Rate).

Instead of pulling the daily annualized percentage rate, we pull the **SOFR Index** (`SOFRINDEX`). The index value acts exactly like a stock price, compounding the daily overnight rates. This makes it incredibly easy to calculate daily financing costs using the exact same `pct_change()` math we use for our futures and equity benchmarks.

In the cell below, we dynamically extract the start and end dates from our futures dataset so we only download the exact SOFR timeframe we need from FRED.

In [ ]:
# 1. Dynamically grab the start and end dates from our existing futures data
start_date = df_futures.index[0]
end_date = df_futures.index[-1]

# 2. Fetch the SOFR Index from FRED for that exact time window
sofr_index = fred_data.get_series('SOFRINDEX',
                                  start_date=start_date,
                                  end_date=end_date)

# 3. Standardize the date format to ensure clean merging later
sofr_index.index = pd.to_datetime(sofr_index.index)

# Display the series
sofr_index

## 🧩 The Final Merge: Bringing It All Together
With our SOFR data fetched, we are ready to assemble the final master dataset.

We use another left join to append the SOFR index to our `df_futures_cash` dataframe. Just as it did with the target indices, the left join ensures the daily SOFR value is correctly duplicated across our roll dates.

Notice that we immediately append `.dropna()` to the merge. This is a critical defensive step: bond markets (which determine SOFR) and equity markets occasionally have mismatched holiday schedules. Dropping missing values here guarantees our matrix is perfectly solid before we calculate our final excess returns.

In [ ]:
# 1. Merge SOFR into the master dataframe and drop any mismatched holiday rows
df_futures_cash_sofr = (df_futures_cash.join(sofr_index, how='left')).dropna()

# 2. Standardize the column name for cleaner code later
df_futures_cash_sofr.rename(columns={'SOFRINDEX': 'sofr_index'}, inplace=True)

# 3. Preview the assembled raw data

# 4. Safeguard: Remove duplicate columns if the cell is run multiple times
df_futures_cash_sofr = df_futures_cash_sofr.loc[:, ~df_futures_cash_sofr.columns.duplicated()]

# Display the roll date to verify
df_futures_cash_sofr.head()

## 🧼 Calculating and Cleaning Percentage Changes
Now we calculate the percentage changes for our entire master dataset—our toolkit futures, benchmark indices, and the SOFR index—in one sweep.

Because we used a left join earlier, our target indices and SOFR data safely duplicated across the roll dates without causing calculation errors. We still, however, have to handle the massive, artificial price gap in the futures contracts themselves. But we already know how to do this with the neat Pandas trick.

Just as we did when caclulating the correlation of the futures, the Pandas `duplicated(keep='first')` trick safely slices out the fake returns created by the roll gap. To verify the cleanup, the code below inspects a known roll date (June 12, 2025) across both raw and cleaned datasets—demonstrating how the second-row gap return is eliminated.


In [ ]:
# 1. Calculate percentage changes across the entire master dataframe
df_pct_dup = df_futures_cash_sofr.pct_change()

# 2. Slice out the fake roll gap returns by keeping only the first instance of each date
df_pct_clean = df_pct_dup[~df_pct_dup.index.duplicated(keep='first')]

# 3. Display the before-and-after for a known roll date to verify
display(df_pct_dup.loc['2025-06-12'])
display(df_pct_clean.loc['2025-06-12'])

## ⚖️ Calculating Excess Returns: The Cost of Carry

As we know from the cost of carry equation, futures returns and cash index returns are not on the same playing field.

* **The Futures ($X$ matrix):** Futures pricing naturally embeds the financing rate (the cost of carry). Because you do not put up the full cash amount to buy a futures contract, the return you generate is inherently an "excess" return. We leave our toolkit futures exactly as they are.

* **The Benchmarks ($y$ vectors):** Our target indices are Total Return indices--a fully funded cash position. To create an apples-to-apples comparison with our futures, we must approximate their funding cost by subtracting the SOFR base rate from the index returns.

In the code below, we pass our futures returns through as-is, and use the Pandas `.sub()` method to neatly subtract the daily SOFR return from each of our benchmark indices across every row.


---


<details>
<summary><b>📝 A Quick Example: Pandas sub() method</b></summary>
<br>

```python
import pandas as pd

# Create a sample DataFrame
df = pd.DataFrame({
    'A':,
    'B': [100, 200, 300]})

# 1. Subtract a single number from the entire DataFrame
result_scalar = df.sub(5)

# 2. Subtract a column (Series) from another column
result_column = df['B'].sub(df['A'])

print("--- Original df ---")
print(df)
print("\n--- df.sub(5) ---")
print(result_scalar)
print("\n--- df['B'].sub(df['A']) ---")
print(result_column)


# --- Original df ---
#    A    B
# 0 10  100
# 1 20  200
# 2 30  300

# --- df.sub(5) ---
#    A    B
# 0  5   95
# 1 15  195
# 2 25  295

# --- df['B'].sub(df['A']) ---
# 0     90
# 1    180
# 2    270
dtype: int64
```
</details>

In [ ]:
# 1. Copy the futures returns as-is (they already embed financing costs)
df_excess_returns = df_pct_clean[['SP500_FUT', 'RUSS_FUT', 'NSDQ_FUT', 'DJIA_FUT']].copy()

# 2. Define the lists of our raw target columns and our new clean column names
raw_portfolios = ['SP500_tr', 'SPMID_tr', 'SPSML_tr', 'SPGRW_tr', 'SPVAL_tr']
excess_portfolios = ['SP500', 'SPMID', 'SPSMALL', 'SPVALUE', 'SPGROWTH']

# 3. Subtract daily SOFR from the total return indices to calculate true excess returns
df_excess_returns[excess_portfolios] = df_pct_clean[raw_portfolios].sub(df_pct_clean['sofr_index'], axis=0)

# Preview the final, fully aligned excess return dataset
df_excess_returns.dropna().head()

## 🧮 Running the Spanning Regressions
We have finally reached the core of the exercise. We'll use the clean, perfectly aligned excess returns to calculate exactly how much of each toolkit contract we need to replicate our target portfolios.

To do this, we run an Ordinary Least Squares (OLS) regression for each of our five targets.

**A Quick Callback to Statsmodels (`sm`):**
As discussed earlier, statsmodels is our luxury diagnostic tool. We don't need to worry about the deep matrix calculus under the hood; we use statsmodels because it neatly outputs the exact financial metrics we care about—allocation weights, standard errors, and $R^2$—right out of the box.

*   **The Dependent Variable ($y$):** The excess return of the target index.
*   **The Independent Variables ($X$):** The returns of our four liquid futures contracts.

**The Intercept ("Delta Funding"):**

Notice that we add a constant to our regression using sm.add_constant(), but we rename it to "Delta Funding" instead of constant or intercept. In synthetic replication, this parameter represents the residual funding spread: the difference between the theoretical financing cost we subtracted (SOFR) and the actual financing cost embedded in the futures market. As demonstrated in the previous notebook, implied futures funding is bound by arbitrage channels. Furthermore, while SOFR anchors Treasury financing, equity financing rates can differ. Between this spread and the fact that our spanning portfolio cannot capture 100% of the market risk perfectly, we expect this constant to deviate from zero.

> **💡 Keep in Mind: The "Unspanned" Penalty**
>
> **The S&P 500 is the only target portfolio with a direct, matching contract in our toolkit. For the other target portfolios, our four futures will struggle to capture the excess returns perfectly. Consequently, the intercept stops being a pure measure of "Delta Funding" and begins to absorb the return of those unspanned risks. Furthermore, because we are running a multiple regression, even the S&P 500's intercept will be slightly contaminated by the inclusion of the other three competing futures contracts.**

**The Output:**
The regression coefficients provide our exact asset allocation weights. We will extract these weights alongside their standard errors (to gauge stability) and the $R^2$ (to measure exactly how much of the target's daily variance we successfully captured).

In [ ]:
# 1. Isolate the independent variables (X matrix) and drop any remaining missing values
futures = df_excess_returns[['SP500_FUT', 'RUSS_FUT', 'NSDQ_FUT', 'DJIA_FUT']].dropna()

# 2. Define the dependent variables (y vectors) we want to replicate
portfolios = ['SP500', 'SPMID', 'SPSMALL', 'SPVALUE', 'SPGROWTH']

# Dictionary to store the results for each portfolio
allocations = {}

# 3. Loop through each target portfolio and run the OLS regression
for port in portfolios:
    # Set the target y vector
    y = df_excess_returns[port].dropna()

    # Set the X matrix and add the intercept (renamed to Delta Funding)
    x = sm.add_constant(futures.dropna()).rename(columns={'const': 'Delta Funding'})

    # Fit the OLS model
    model = sm.OLS(y, x)
    results = model.fit()

    # Extract the weights (params) and standard errors (bse) into a clean table
    df = pd.DataFrame([results.params[0:], results.bse[0:]],
                      index=['Allocation', 'Std. Error'])

    # Add the R-squared value to measure replication success
    df['R-squared'] = [results.rsquared, np.nan]

    # Save to our dictionary
    allocations[port] = df

# 4. Combine all results into a single, highly readable master table
df_allocations = pd.concat(allocations).fillna('')
df_allocations.index.names = ['Portfolio', 'Metric']
df_allocations

## 📈 Interpreting the Weights: The Beauty of Futures Plumbing

When you look at the `Allocation` row in the table above, you might notice something interesting: some of the weights are negative. In the math of our OLS regression, a negative coefficient simply represents a short position.

If we were trying to replicate these target benchmarks in the cash market (using physical stocks or ETFs), negative weights introduce additional friction. While shorting highly liquid assets is relatively simple, cash shorting always requires borrowing shares. This means navigating capital-intensive margin rules, paying variable stock borrow fees, and accepting the risk that borrowed shares could be recalled. Often, quantitative models produce theoretical cash portfolios whose short positions drag down real-world performance due to these borrow costs.

This is where we can sing the praises of the futures market:

*   **Structural Symmetry:** In futures, there is no mechanical difference between going long and going short. You are simply taking one side of a contractual agreement.
*   **No Borrowing Required:** You never have to locate shares or pay stock borrow fees. The exchange clearinghouse handles all the complex plumbing for you.
*   **Actionable Math:** Because futures neutralize the costs and mechanics of short-selling, we did not have to artificially constrain our regression to be "long-only." The raw, unconstrained weights our model produced can be efficiently executed in a live trading environment.

## 🔍 What About Delta Funding?
Notice that the S&P 500 Delta Funding estimate is 0.000024 (or 0.0024% per day). Compounding that over 252 trading days results in an annualized estimate of about 61 basis points.

You might notice that this number is less than the total tracking error we calculated in the previous notebook. This is a live demonstration of the "contamination" we warned about earlier. Because we are running a multiple regression, the other contracts—particularly the Nasdaq 100—are carrying part of the load. In essence, some of the true arbitrage band variation is being absorbed by the other coefficients rather than sitting purely in the intercept.

## 🔮 The Reality Check: Out-of-Sample Testing
Up to this point, our regression has looked at the entire dataset at once. Analyzing in-sample $R^2$ is helpful for understanding historical relationships, but it suffers from a massive flaw: hindsight is 20/20. In the real world, we do not have tomorrow's data to calculate today's weights.

To prove our spanning toolkit actually works, we need an out-of-sample test. We will use a one-step-ahead rolling window:

1. **Train:** Look at exactly 122 historical trading days (roughly 6 months) to calculate our optimal asset allocation weights.
2. **Predict:** Apply those weights to the very next trading day's futures returns to generate a synthetic portfolio return.
3. **Roll:** Move the window forward by one day and repeat the process.

**Tuning the Model: Memory and Market Regimes**

In the function below, we have to make two critical decisions about how our model learns:
*   **`NOBS=122` (Number of Observations):** We default to 122 days, which is roughly six months of trading data. Two front-month contracts provide enough data to shrink our standard errors, but are short enough so that our weights reflect the *current* market environment.
*   **`ROLL=True` (Rolling vs. Expanding Window):** Market correlations are not static; they shift over time. By using a *rolling* window, we drop the oldest day of data every time we move forward. This allows the model to "forget" stale relationships and adapt dynamically to new market regimes. If we set this to `False`, it would use an *expanding* window, assuming that the very first day of data is just as relevant today as it was years ago.

**A Quick Note on the Code:**
Because we are about to calculate regressions in a loop hundreds of times, using `statsmodels` would be incredibly slow. In the `predict_actual` function below, we trade our luxury dashboard for the stripped-down race car: pure C-compiled NumPy matrix algebra. It is the exact same OLS math, just stripped of the statistical summaries so it can run at lightning speed.


In [ ]:
def predict_actual(portfolio_all, futures_all, NOBS=122, ROLL=True):
    """
    Performs a 1-step ahead out-of-sample forecast using OLS matrix algebra.
    """
    y = portfolio_all.to_numpy()

    # Create a copy so we don't mutate the original DataFrame
    futures_inputs = futures_all.copy()

    # Add our 'Delta Funding' intercept column (constant of 1.0)
    futures_inputs.insert(0, 'Delta Funding', 1.0)
    x = futures_inputs.to_numpy()

    # Initialize arrays to hold our daily results with empty (NaN) values
    forecast = np.full(len(portfolio_all), np.nan)
    actual = np.full(len(portfolio_all), np.nan)

    # Loop through the timeline, starting after our initial 122-day training window
    for date in range(NOBS, len(portfolio_all)):

        # Isolate the training data
        if ROLL: # Rolling window (drops oldest data as it moves)
            y_estimate = y[date-NOBS:date]
            x_estimate = x[date-NOBS:date]
        else:    # Expanding window (keeps all historical data)
            y_estimate = y[:date]
            x_estimate = x[:date]

        # Calculate the OLS allocation weights using pure linear algebra for speed
        future_allocations = np.linalg.solve(x_estimate.T @ x_estimate, x_estimate.T @ y_estimate)

        # Apply the calculated weights to today's actual futures returns
        forecast[date] = future_allocations @ x[date]
        actual[date] = y[date]

    return forecast, actual

## 📉 Visualizing the Drift: Tracking Error
If our synthetic portfolio perfectly replicated the target benchmark, the daily difference between our forecast and the actual excess return would be exactly zero. In reality, there is always friction, residual noise, and shifting correlations. We call this difference **Tracking Error**.

To see how this error behaves over time, we will compound it cumulatively. A flat tracking error line means our synthetic replication is perfectly mirroring the target. A line drifting sharply up or down means our model is structurally bleeding performance.

In [ ]:
def plot_tracking_error(actual_forecast, portfolio_name):
    """
    Plots the cumulative tracking error between the actual benchmark
    and the synthetic futures portfolio.
    """
    plot_data = [actual_forecast]
    series = [f"{portfolio_name} Actual Less Forecast"]
    x_axis = actual_forecast.index
    size = (7, 5)
    colors = ['b']

    # Use nanmin and nanmax so that NaN values are safely ignored when scaling
    y_lower = np.nanmin(plot_data) - 0.002
    y_upper = np.nanmax(plot_data) + 0.002
    ylim = [y_lower, y_upper]

    markers = ['']
    title = f"{portfolio_name} Actual Return Less Predicted From Futures"
    xlabel = 'Trading Date'
    ylabel = 'Cumulative Tracking Error'

    fq.one_y_axis(
        x_axis,
        plot_data,
        title=title,
        series_labels=series,
        xlabel=xlabel,
        y_limits=ylim,
        ylabel=ylabel,
        markers=markers,
        colors=colors,
        figure_size=size
    )

## ⚙️ Executing the S&P 500 Forecast
To visualize this drift, the code below executes our rolling one-step-ahead forecast for the S&P 500:

* **Isolate Variables:** We define the S&P 500 cash returns as our target ($y$) and the four futures contracts as our toolkit ($X$).
*  **Predict & Compare:** We run the rolling regression and calculate the daily tracking error (actual - forecast).
* **Compound & Plot:** We compound the error cumulatively (cumprod) to reveal the structural deviation.

In [ ]:
df_y=df_excess_returns['SP500'].dropna()
df_x=df_excess_returns[['SP500_FUT','RUSS_FUT','NSDQ_FUT','DJIA_FUT']].dropna()
forecast,actual=predict_actual(df_y,df_x)
actual_forecast=(1+pd.Series(actual-forecast,index=df_y.index)).cumprod()
plot_tracking_error(actual_forecast,portfolio_name='SP500')

### 📉 S&P 500 Tracking Error: A Callback to Arbitrage Bands

The tracking error plot for the S&P 500 reveals a distinct dip around mid-2026, marking a period where the actual cash index underperformed our synthetic futures forecast.

This deviation perfectly mirrors the arbitrage band dynamics we explored in the previous chapter. The sudden spike in general market uncertainty likely distorted the pricing dynamics between the contracts.

<details>
  <summary><b>💡 Click here to view the Arbitrage Band & VIX chart from the previous chapter</b></summary>
  <br>
  <i><b>Reading the Chart:</b> Because the tracking error line plots Actual minus Forecast, a downward slope means our synthetic futures portfolio is actively <b>outperforming</b> the cash benchmark.</i>
  <br><br>
  <img src="https://raw.githubusercontent.com/PatrickJHess/Static_Data_Repo/main/images/arbbands.jpeg" alt="Arbitrage Bands and VIX" width="800"/>
</details>

## 📏 The Size Challenge: Spanning MidCap and SmallCap

Our core futures toolkit seamlessly captures the massive liquidity and large-cap beta of the S&P 500. Not a breathtaking moment--the S&P contract related directly to the index. As we move down the market capitalization spectrum we'll see larger deviations.

The Russell 2000 (RUSS_FUT) provides a direct anchor for small-cap exposure, but the MidCap space (SPMID) sits in a liquidity "valley." It lacks a dedicated, highly liquid futures contract in our macro toolkit, forcing the OLS regression to dynamically blend large-cap and small-cap futures to approximate the exposure.

What to watch for: As you run the code below, pay close attention to the y-axis. You will see wider tracking error bounds compared to the remarkably tight S&P 500 plot. This visualizes the structural reality of synthetic replication: macro index futures cannot perfectly isolate the idiosyncratic risks of smaller companies.

In [ ]:
# 1. Evaluate MidCap (SPMID)
df_y_mid = df_excess_returns['SPMID'].dropna()
df_x_mid = df_excess_returns[['SP500_FUT', 'RUSS_FUT', 'NSDQ_FUT', 'DJIA_FUT']].dropna()

forecast_mid, actual_mid = predict_actual(df_y_mid, df_x_mid)
actual_forecast_mid = (1 + pd.Series(actual_mid - forecast_mid, index=df_y_mid.index)).cumprod()
plot_tracking_error(actual_forecast_mid, portfolio_name='SPMID (MidCap)')

# 2. Evaluate SmallCap (SPSML)
df_y_sml = df_excess_returns['SPSMALL'].dropna()
df_x_sml = df_excess_returns[['SP500_FUT', 'RUSS_FUT', 'NSDQ_FUT', 'DJIA_FUT']].dropna()

forecast_sml, actual_sml = predict_actual(df_y_sml, df_x_sml)
actual_forecast_sml = (1 + pd.Series(actual_sml - forecast_sml, index=df_y_sml.index)).cumprod()
plot_tracking_error(actual_forecast_sml, portfolio_name='SPSML (SmallCap)')

### 📻 A Note on the Mid-2026 Dip (Signal vs. Noise)
You will notice that the VIX-driven tracking error dip in mid-2026 is still present in both Size benchmarks. In absolute terms, the deviation is actually quite large. But relative to the wider y-axis bounds, it visually blends into the rest of the chart.   This perfectly illustrates a classic signal-to-noise problem in quantitative replication. With the S&P 500, our baseline replication was so tight (low noise) that the structural shift (the signal) stood out vividly. For MidCap and SmallCap, our macro toolkit naturally produces a much noisier baseline tracking error. That massive underlying variance drowns out the arbitrage bands, making it significantly harder to distinguish a structural pricing dislocation from standard replication drift.

## ⚖️ The Style Challenge: Spanning Growth vs. Value
Finally, we turn our attention to the style factors: Growth (`SPGROWTH`) and Value (`SPVALUE`).

Our independent variables include the tech-heavy Nasdaq 100 (`NSDQ_FUT`) and the industrials/financials-heavy Dow Jones (`DJIA_FUT`).

What to watch for: In theory, Growth's concentration in mega-cap tech maps fairly closely to the Nasdaq 100, and Value's exposure to idiosyncratic sectors (like regional banks, energy, and utilities) relies on a looser blend of our macro toolkit. As you run the code below, pay attention to whether these structural assumptions hold up out-of-sample, particularly as market regimes shift later in the year.

In [ ]:
# 3. Evaluate Growth (SPGRW)
df_y_grw = df_excess_returns['SPGROWTH'].dropna()
df_x_grw = df_excess_returns[['SP500_FUT', 'RUSS_FUT', 'NSDQ_FUT', 'DJIA_FUT']].dropna()

forecast_grw, actual_grw = predict_actual(df_y_grw, df_x_grw)
actual_forecast_grw = (1 + pd.Series(actual_grw - forecast_grw, index=df_y_grw.index)).cumprod()
plot_tracking_error(actual_forecast_grw, portfolio_name='SPGRW (Growth)')

# 4. Evaluate Value (SPVAL)
df_y_val = df_excess_returns['SPVALUE'].dropna()
df_x_val = df_excess_returns[['SP500_FUT', 'RUSS_FUT', 'NSDQ_FUT', 'DJIA_FUT']].dropna()

forecast_val, actual_val = predict_actual(df_y_val, df_x_val)
actual_forecast_val = (1 + pd.Series(actual_val - forecast_val, index=df_y_val.index)).cumprod()
plot_tracking_error(actual_forecast_val, portfolio_name='SPVAL (Value)')

### 📊 Analyzing the Style Results: A Regime Shift

At first glance, these results challenge standard expectations. Growth, due to its tech-heavy futures alignment, was expected to replicate more cleanly than Value. But the out-of-sample data reveals a fascinating regime shift in the back half of 2026:

* **The Growth Whipsaw:** Following the mid-2026 volatility spike, the tracking error for Growth does not recover cleanly; instead, it becomes extremely volatile, experiencing massive swings through the end of the year. As implied volatility collapses post-spike, intense sector rotation within high-beta tech causes individual stock dispersion to explode—temporarily decoupling the predictive relationship between Nasdaq futures and S&P Growth benchmark.
* **The Value Stabilization:** Conversely, the Value factor stabilizes after the mid-2026 blip. While it suffered a steady drift earlier in the year, the back half shows a remarkably stable, sideways tracking error. As macro uncertainty subsides, low-beta cash-flow drivers return to steady-state behavior, allowing our futures toolkit to "find its footing."

Both behaviors are entirely consistent with the aftermath of a VIX regime break: the sudden collapse in broader market uncertainty calms value factor dispersion while triggering aggressive, unpredictable factor rotations within growth.

This is the ultimate lesson of out-of-sample testing: static assumptions do not survive dynamic markets. Regimes shift, correlations break, and a spanning toolkit that perfectly tracks a factor in one environment can suddenly struggle when market volatility regime-shifts.

## 🎓 Key Takeaways & Synthesis

1. **The Core Mechanism: Cost-of-Carry & Theoretical Arbitrage**

   At its foundation, synthetic replication relies on the cost-of-carry model: $$F_t = S_t\times\left(1+(r_t+\Delta_t)\times\frac{T-t}{360}\right) - FVD_t.$$
   In frictionless, low-volatility markets, index futures returns track their spot benchmarks with near-zero residual risk. The OLS rolling regression acts as a dynamic hedge ratio ($X \beta \approx y$), constantly rebalancing weights to span the benchmark's systematic risk factors.

2. **Microstructure Disruptions & Arbitrage Bands**

   Real-world markets do not operate without friction. During periods of heightened macro uncertainty (marked by VIX spikes), balance sheets tight putting the brakes on arbing expensive futures prices. Because execution is bound by arbitrage bands—transaction costs, shorting constraints, and capital costs—arbitrageurs only step in to restore equilibrium once the dislocation exceeds these boundary limits, creating observable, temporary dips in cumulative tracking error ($y_t - \hat{y}_t$).

3. **Spanning Granularity Across Size & Style**

   * **Size (MidCap & SmallCap):** Moving down the market-cap spectrum introduces a classic signal-to-noise challenge. Without a perfectly liquid, bespoke futures contract for every sub-index, the macro toolkit produces higher baseline variance (noise). While absolute dislocations remain large during market shocks, this elevated baseline error makes arbitrage bands harder to isolate visually.
   * **Style (Growth vs. Value):** It might be expected that Growth is easier to span due to the strong alignment between Nasdaq futures and mega-cap tech, out-of-sample data reveals its vulnerability to post-volatility regime shifts. As VIX normalizes, factor rotations and individual stock dispersion within high-beta growth disrupt historical factor loadings. Conversely, low-beta, cash-flow-anchored Value stocks stabilize post-shock, allowing the futures toolkit to regain its tracking precision.

**The Bottom Line:** Out-of-sample tracking error is not merely a diagnostic tool for statistical fit—it is a live reflection of market microstructure, derivative pricing bounds, and macro regime transitions. Robust portfolio design requires evaluating replication models across both calm and stressed market regimes.

## 🥊 Stepping Into the Ring: The Capstone Challenge

We have established the theoretical foundation of synthetic replication and studied the mechanics of market frictions and tracking error. But hitting the heavy bag in the gym is entirely different from a live bout. The true test of a quantitative model is how well it can take a punch out-of-sample.

In this final challenge, you will step out of the guided examples and build a replication model from the ground up. Your mission is to construct the data pipeline, strip out funding rates, and deploy our four-contract toolkit to go toe-to-toe with two highly distinct sector portfolios: **S&P 500 Financials** and **S&P 500 Information Technology**.

<div style="background-color: #fff3cd; color: #856404; border-left: 5px solid #ffeeba; padding: 15px; margin: 20px 0; border-radius: 4px; font-family: sans-serif;">
  <strong>⚠️ Corner Advice:</strong> Take your time. This is a full heavyweight bout, not a quick sparring session. Focus on your technique when building the data pipeline and the results will follow.
</div>

Choose your arena below to lace up your gloves and begin the challenge:

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-top: 20px;">

<a href="https://patrickjhess.github.io/Derivatives-Futures-Chapter-Three/chapter-challenge" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 32px; box-sizing: border-box;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/3/38/Jupyter_logo.svg" alt="Jupyter Logo" style="height: 18px; margin-right: 8px;"> Futures Volume: Chapter Three
</a>
    <a href="https://colab.research.google.com/github/PatrickJHess/Derivatives-Futures-Chapter-Three/blob/master/colab/Colab_Chapter_Challenge.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 32px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Derivatives-Futures-Chapter-Three/master?urlpath=lab/tree/notebooks/Chapter_Challenge.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 32px; box-sizing: border-box;">
    Launch Live in Binder
  </a>


</div>